In [55]:
#imports
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
import random
layers = tf.keras.layers
Model = tf.keras.Model

In [56]:
# # Create a folder in your writable working directory
!mkdir -p /kaggle/working/cub200

# # Extract the .tgz file from the input directory into the working directory
!tar -xzf /kaggle/input/datasets/hridayeshrana/cub-200-2011/CUB_200_2011.tgz -C /kaggle/working/cub200

In [57]:

print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.20.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [58]:
print(tf.config.experimental.list_physical_devices('GPU'))
import subprocess
print(subprocess.run(['nvidia-smi', 'topo', '-m'], capture_output=True, text=True).stdout)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks



In [59]:
strategy = tf.distribute.MirroredStrategy(cross_device_ops=tf.distribute.HierarchicalCopyAllReduce())
print(f"Number of devices: {strategy.num_replicas_in_sync}")

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of devices: 2


In [60]:
def load_and_resize(path1, path2, label):
    img1 = tf.io.read_file(path1)
    img2 = tf.io.read_file(path2)
    img1 = tf.image.decode_image(img1, channels=3, expand_animations=False)
    img2 = tf.image.decode_image(img2, channels=3, expand_animations=False)
    img1 = tf.image.resize(img1, (224, 224))
    img2 = tf.image.resize(img2, (224, 224))
    img1 = tf.cast(img1, tf.float32) / 255.0
    img2 = tf.cast(img2, tf.float32) / 255.0
    return (img1, img2), label

def augment_img(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.image.random_saturation(img, lower=0.85, upper=1.15)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img

def augment_pair(imgs, label):
    img1, img2 = imgs
    img1 = augment_img(img1)
    img2 = augment_img(img2)
    return (img1, img2), label

def create_dataset(pairs, batch_size=8, training=False):
    img1_paths = [p[0] for p in pairs]
    img2_paths = [p[1] for p in pairs]
    labels     = [p[2] for p in pairs]
    dataset = tf.data.Dataset.from_tensor_slices(
        ((img1_paths, img2_paths), labels)
    )
    options = tf.data.Options()
    options.experimental_deterministic = False
    dataset = dataset.with_options(options)

    # Shuffle paths/labels BEFORE decode — cheap (strings + ints), not full images
    dataset = dataset.shuffle(buffer_size=len(pairs), reshuffle_each_iteration=True)

    dataset = dataset.map(
        lambda paths, label: load_and_resize(paths[0], paths[1], label),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    # no .cache() — pairs regenerate every epoch, dataset is single-use

    if training:
        dataset = dataset.map(augment_pair, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size, drop_remainder=True)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [61]:
import random

def make_pairs(char_dict, num_pairs=20):
    class_names = list(char_dict.keys())
    all_pairs = []
    for current_class in class_names:
        images = char_dict[current_class]
        if len(images) < 2:
            continue  # skip species with too few images to form a same-pair

        # same pairs (label = 1)
        used_same = set()
        attempts = 0
        max_attempts = num_pairs * 20  # safety valve for small image counts
        while len(used_same) < num_pairs and attempts < max_attempts:
            img1 = random.choice(images)
            img2 = random.choice(images)
            attempts += 1
            if img2 == img1:
                continue
            pair_check = tuple(sorted([img1, img2]))
            if pair_check in used_same:
                continue
            used_same.add(pair_check)
            all_pairs.append([img1, img2, 1])

        # different pairs (label = 0), drawing class2 from a wide pool each time
        used_diff = set()
        attempts = 0
        while len(used_diff) < num_pairs and attempts < max_attempts:
            class2 = random.choice([x for x in class_names if x != current_class])
            img1 = random.choice(images)
            img2 = random.choice(char_dict[class2])
            attempts += 1
            pair_check = tuple(sorted([img1, img2]))
            if pair_check in used_diff:
                continue
            used_diff.add(pair_check)
            all_pairs.append([img1, img2, 0])

    random.shuffle(all_pairs)
    return all_pairs

In [62]:
class L2Normalize(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

class EuclideanDistance(tf.keras.layers.Layer):
    def call(self, tensors):
        return tf.norm(tensors[0] - tensors[1], axis=1)

In [63]:
layers = tf.keras.layers
Model = tf.keras.Model
regularizers = tf.keras.regularizers

def build_encoder(input_shape=(224, 224, 3), embedding_dim=128, freeze_backbone=False, l2_reg=1e-4):
    inputs = layers.Input(shape=input_shape, name="image")
    base_model = tf.keras.applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape,
        pooling='avg'
    )
    base_model.trainable = not freeze_backbone
    x = base_model(inputs)
    x = layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(embedding_dim, kernel_regularizer=regularizers.l2(l2_reg))(x)
    embeddings = L2Normalize()(x)
    return Model(inputs, embeddings, name='encoder')

def build_siamese_model(enc):
    img1 = layers.Input(shape=(224, 224, 3), name='img1')
    img2 = layers.Input(shape=(224, 224, 3), name='img2')
    tensor1 = enc(img1)
    tensor2 = enc(img2)
    distance = EuclideanDistance()([tensor1, tensor2])
    return Model([img1, img2], distance)

In [64]:
import os
import random
from sklearn.model_selection import train_test_split

def load_and_split_cub_images(images_dir, test_size=0.2, val_size=0.1):
    """
    Loads CUB-200 images and splits species into train/val/test sets.
    val_size is the fraction of the *original* total reserved for validation
    (taken out of the remaining train portion after the test split).
    """
    all_species = [d for d in os.listdir(images_dir) if os.path.isdir(os.path.join(images_dir, d))]

    # First split off the test species
    train_val_species, test_species = train_test_split(
        all_species, test_size=test_size, random_state=42
    )

    # Then split the remainder into train/val
    # Adjust val_size so it represents the right fraction of the original whole
    relative_val_size = val_size / (1 - test_size)
    train_species, val_species = train_test_split(
        train_val_species, test_size=relative_val_size, random_state=42
    )

    def get_images_for_species(species_list):
        data = {}
        for species in species_list:
            species_path = os.path.join(images_dir, species)
            full_paths = [
                os.path.join(species_path, f)
                for f in os.listdir(species_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg'))
            ]
            if full_paths:
                data[species] = full_paths
        return data

    return (
        get_images_for_species(train_species),
        get_images_for_species(val_species),
        get_images_for_species(test_species),
    )

# 1. Load and Split
cub_path = '/kaggle/working/cub200/CUB_200_2011/images'
train_dict, val_dict, test_dict = load_and_split_cub_images(cub_path, test_size=0.2, val_size=0.1)
print(f"Species for training: {len(train_dict)}, validation: {len(val_dict)}, testing: {len(test_dict)}")

# 2. Generate Pairs
train_pairs = make_pairs(train_dict, num_pairs=40)
val_pairs = make_pairs(val_dict, num_pairs=10)
test_pairs = make_pairs(test_dict, num_pairs=10)  # Fewer pairs for testing

# 3. Create Datasets
train_dataset = create_dataset(train_pairs, batch_size=16, training=True)
val_dataset = create_dataset(val_pairs, batch_size=16, training=False)
test_dataset = create_dataset(test_pairs, batch_size=16, training=False)

# 4. Distribute
train_dataset = strategy.experimental_distribute_dataset(train_dataset)
val_dataset = strategy.experimental_distribute_dataset(val_dataset)
test_dataset = strategy.experimental_distribute_dataset(test_dataset)

print(f"Ready: {len(train_pairs)} training pairs, {len(val_pairs)} validation pairs, {len(test_pairs)} test pairs.")

Species for training: 140, validation: 20, testing: 40
Ready: 11200 training pairs, 400 validation pairs, 800 test pairs.


In [65]:
with strategy.scope():
    enc     = build_encoder()
    siamese = build_siamese_model(enc)
    siamese.load_weights('/kaggle/working/best_model.weights.h5')
    enc = siamese.get_layer('encoder')

    initial_lr = 1e-4
    optimizer  = tf.keras.optimizers.Adam(initial_lr)
    enc.summary()
    siamese.summary()

Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 2048)           │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ l2_normalize_3 (L2Normalize)    │ (None, 128)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,801,024 (94.61 MB)

 Trainable params: 24,747,904 (94.41 MB)

 Non-trainable params: 53,120 (207.50 KB)

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ img1 (InputLayer)   │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ img2 (InputLayer)   │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder             │ (None, 128)       │ 24,801,024 │ img1[0][0],       │
│ (Functional)        │                   │            │ img2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ euclidean_distance… │ (None)            │          0 │ encoder[0][0],    │
│ (EuclideanDistance) │                   │            │ encoder[1][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,801,024 (94.61 MB)

 Trainable params: 24,747,904 (94.41 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [66]:
def compute_embeddings(encoder, char_dict, sample_per_class=8):
    cache = {}
    for class_name, images in char_dict.items():
        sampled = random.sample(images, min(sample_per_class, len(images)))
        for img_path in sampled:
            img = tf.io.read_file(img_path)
            img = tf.image.decode_image(img, channels=3, expand_animations=False)
            img = tf.image.resize(img, (224, 224))
            img = tf.cast(img, tf.float32) / 255.0
            img = tf.expand_dims(img, 0)
            emb = encoder(img, training=False).numpy()[0]
            cache[img_path] = emb
    return cache


def make_pairs_with_hard_negatives(char_dict, embeddings_cache=None,
                                   num_pairs=40, hard_neg_ratio=0.5):
    class_names  = list(char_dict.keys())
    all_pairs    = []
    num_hard     = int(num_pairs * hard_neg_ratio) if embeddings_cache else 0
    num_random   = num_pairs - num_hard
    max_attempts = num_pairs * 20

    for current_class in class_names:
        images = char_dict[current_class]
        if len(images) < 2:
            continue

        # same pairs (label = 1)
        used_same = set()
        attempts  = 0
        while len(used_same) < num_pairs and attempts < max_attempts:
            img1, img2 = random.choice(images), random.choice(images)
            attempts  += 1
            if img1 == img2:
                continue
            key = tuple(sorted([img1, img2]))
            if key in used_same:
                continue
            used_same.add(key)
            all_pairs.append([img1, img2, 1])

        used_diff = set()

        # random negatives (label = 0)
        attempts = 0
        while len(used_diff) < num_random and attempts < max_attempts:
            class2 = random.choice([x for x in class_names if x != current_class])
            img1   = random.choice(images)
            img2   = random.choice(char_dict[class2])
            attempts += 1
            key = tuple(sorted([img1, img2]))
            if key in used_diff:
                continue
            used_diff.add(key)
            all_pairs.append([img1, img2, 0])

        # hard negatives (label = 0)
        if num_hard > 0 and embeddings_cache:
            anchors = [(p, embeddings_cache[p]) for p in images if p in embeddings_cache]
            if not anchors:
                continue

            other_classes   = [x for x in class_names if x != current_class]
            sampled_classes = random.sample(other_classes, min(30, len(other_classes)))
            candidates      = [
                (p, embeddings_cache[p])
                for cls in sampled_classes
                for p   in char_dict[cls]
                if p in embeddings_cache
            ]
            if not candidates:
                continue

            cand_paths = [c[0] for c in candidates]
            cand_embs  = np.stack([c[1] for c in candidates])

            hard_found = 0
            for anchor_path, anchor_emb in anchors:
                if hard_found >= num_hard:
                    break
                dists = np.linalg.norm(cand_embs - anchor_emb, axis=1)
                for idx in np.argsort(dists):
                    key = tuple(sorted([anchor_path, cand_paths[idx]]))
                    if key in used_diff:
                        continue
                    used_diff.add(key)
                    all_pairs.append([anchor_path, cand_paths[idx], 0])
                    hard_found += 1
                    break

    random.shuffle(all_pairs)
    return all_pairs

In [67]:
import numpy as np
from tqdm.auto import tqdm

FREEZE_EPOCHS   = 5     # epochs 0-4: head only, backbone fully frozen
TOTAL_EPOCHS    = 20
WARMUP_EPOCHS   = 3     # hard-neg mining kicks in from here, unchanged

# Staged unfreeze: 5 layers added at each of these epoch boundaries
UNFREEZE_STAGE_EPOCHS = [5, 8, 11, 14]     # 4 stages, 3 epochs apart
LAYERS_PER_STAGE      = 5
STAGE_LRS             = [1e-5, 5e-6, 2e-6, 1e-6]   # progressively gentler as more params unlock

def contrastive_loss(labels, distances, margin=1.0):
    labels = tf.cast(labels, tf.float32)
    same   = labels       * tf.square(distances)
    diff   = (1 - labels) * tf.square(tf.maximum(margin - distances, 0))
    return same + diff

def _train_step(batch):
    (img1, img2), labels = batch
    with tf.GradientTape() as tape:
        distances = siamese([img1, img2], training=True)
        loss = tf.reduce_mean(contrastive_loss(labels, distances))
    grads = tape.gradient(loss, siamese.trainable_variables)
    optimizer.apply_gradients(zip(grads, siamese.trainable_variables))
    return loss

def _val_step(batch):
    (img1, img2), labels = batch
    distances = siamese([img1, img2], training=False)
    return tf.reduce_mean(contrastive_loss(labels, distances))

train_step = tf.function(_train_step)
val_step   = tf.function(_val_step)

def distributed_train_step(batch):
    per_replica = strategy.run(train_step, args=(batch,))
    return strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica, axis=None)

def distributed_val_step(batch):
    per_replica = strategy.run(val_step, args=(batch,))
    return strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica, axis=None)

def unfreeze_next_stage(base_model, n_layers, stage_idx):
    """
    Unfreezes the next `n_layers` non-BatchNorm layers, counting from the
    deepest (last) layers inward. BatchNorm stays frozen throughout —
    fine-tuning BN running stats on small batches (16 here) is a common
    source of instability and isn't worth the risk for this dataset size.
    """
    trainable_layers = [
        l for l in base_model.layers
        if not isinstance(l, tf.keras.layers.BatchNormalization)
    ]
    total_to_unfreeze = n_layers * (stage_idx + 1)
    target_layers = trainable_layers[-total_to_unfreeze:]
    for layer in target_layers:
        layer.trainable = True
    print(f"  Stage {stage_idx + 1}: unfroze last {total_to_unfreeze} non-BN layers "
          f"({[l.name for l in target_layers[-n_layers:]]})")

patience_counter    = 0
train_loss_history  = []
val_loss_history    = []
best_val_loss       = float("inf")
embeddings_cache    = None
current_stage       = -1   # -1 = fully frozen, 0..3 = which stage we're in

print("Starting Training...\n")
print(f"Trainable vars: {len(siamese.trainable_variables)}")
for epoch in range(TOTAL_EPOCHS):
    print(f"Epoch {epoch + 1}/{TOTAL_EPOCHS}")

    # ── Freeze / staged unfreeze ─────────────────────────────────────────
    if epoch == 0:
        enc.get_layer('resnet50').trainable = True   # allow layer-level control below
        for layer in enc.get_layer('resnet50').layers:
            layer.trainable = False                   # start fully frozen
        print("  Backbone: FROZEN (head only)")
    elif epoch in UNFREEZE_STAGE_EPOCHS:
        current_stage += 1
        unfreeze_next_stage(enc.get_layer('resnet50'), LAYERS_PER_STAGE, current_stage)
        new_lr = STAGE_LRS[current_stage]
        with strategy.scope():
            optimizer = tf.keras.optimizers.Adam(new_lr)
        train_step = tf.function(_train_step)   # retrace: new trainable set + new optimizer
        val_step   = tf.function(_val_step)
        print(f"  Backbone: STAGE {current_stage + 1} unfrozen  (LR → {new_lr:.1e})")

    # ── Regenerate pairs every epoch ─────────────────────────────────────
    hard_ratio  = 0.0 if epoch < WARMUP_EPOCHS else 0.5
    train_pairs = make_pairs_with_hard_negatives(
        train_dict, embeddings_cache=embeddings_cache,
        num_pairs=40, hard_neg_ratio=hard_ratio
    )
    val_pairs = make_pairs(val_dict, num_pairs=25)

    train_dataset = create_dataset(train_pairs, batch_size=16, training=True)
    val_dataset   = create_dataset(val_pairs,   batch_size=16, training=False)
    train_dataset = strategy.experimental_distribute_dataset(train_dataset)
    val_dataset   = strategy.experimental_distribute_dataset(val_dataset)
    print(f"  Pairs → {len(train_pairs)} train | {len(val_pairs)} val | hard_ratio={hard_ratio}")
    print(f"  Trainable vars: {len(siamese.trainable_variables)}")

    # ── Train ─────────────────────────────────────────────────────────────
    batch_losses = []
    train_pbar   = tqdm(train_dataset, desc="  Training", unit="batch")
    for batch in train_pbar:
        loss     = distributed_train_step(batch)
        loss_val = loss.numpy()
        batch_losses.append(loss_val)
        train_pbar.set_postfix(loss=f"{loss_val:.4f}")
    avg_train_loss = float(np.mean(batch_losses))
    train_loss_history.append(avg_train_loss)

    # ── Validate ──────────────────────────────────────────────────────────
    val_losses = []
    val_pbar   = tqdm(val_dataset, desc="  Validation", unit="batch")
    for batch in val_pbar:
        val_losses.append(distributed_val_step(batch).numpy())
        val_pbar.set_postfix(loss=f"{val_losses[-1]:.4f}")
    avg_val_loss = float(np.mean(val_losses))
    val_loss_history.append(avg_val_loss)

    print(f"  Summary → Train: {avg_train_loss:.6f} | Val: {avg_val_loss:.6f}")

    # ── Checkpoint + LR reduction ─────────────────────────────────────────
    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        patience_counter = 0
        siamese.save_weights("/kaggle/working/best_model.weights.h5")
        print("  -> Improvement! Saved best model.")
    else:
        patience_counter += 1
        print(f"  -> No improvement ({patience_counter}/3)")
        if patience_counter >= 3:
            new_lr = float(optimizer.learning_rate.numpy()) * 0.5
            optimizer.learning_rate.assign(new_lr)
            patience_counter = 0
            print(f"  -> Reducing LR to {new_lr:.2e}")

    # ── Update embedding cache ────────────────────────────────────────────
    if epoch >= WARMUP_EPOCHS - 1:
        print("  Computing embeddings for hard-neg mining...")
        embeddings_cache = compute_embeddings(enc, train_dict, sample_per_class=8)

print("\nTraining complete.")

Starting Training...

Trainable vars: 218
Epoch 1/20
  Backbone: FROZEN (head only)
  Pairs → 11200 train | 1000 val | hard_ratio=0.0
  Trainable vars: 6


  Training: 0batch [00:00, ?batch/s]

INFO:tensorflow:batch_all_reduce: 6 all-reduces with algorithm = hierarchical_copy, num_packs = 1
INFO:tensorflow:batch_all_reduce: 6 all-reduces with algorithm = hierarchical_copy, num_packs = 1


  Validation: 0batch [00:00, ?batch/s]

  Summary → Train: 0.167711 | Val: 0.137382
  -> Improvement! Saved best model.
Epoch 2/20
  Pairs → 11200 train | 1000 val | hard_ratio=0.0
  Trainable vars: 6


...

  Validation: 0batch [00:00, ?batch/s]

  Summary → Train: 0.174311 | Val: 0.140860
  -> No improvement (1/3)
  Computing embeddings for hard-neg mining...
Epoch 20/20
  Pairs → 9520 train | 1000 val | hard_ratio=0.5
  Trainable vars: 22


  Training: 0batch [00:00, ?batch/s]

  Validation: 0batch [00:00, ?batch/s]

  Summary → Train: 0.172900 | Val: 0.139373
  -> No improvement (2/3)
  Computing embeddings for hard-neg mining...

Training complete.


In [68]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_loss_history) + 1), train_loss_history, label="Train Loss")
plt.plot(range(1, len(val_loss_history) + 1), val_loss_history, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss 2")
plt.legend()
plt.grid(True)
plt.savefig(
    "/kaggle/working/val_vs_train.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

In [69]:
import json

# Extract just the species names (the keys) for the split record
split_record = {
    "train_species": list(train_dict.keys()),
    "val_species": list(val_dict.keys()),
    "test_species": list(test_dict.keys())
}

# Save to a JSON file in your working directory
with open("/kaggle/working/cub200_dataset_splits.json", "w") as f:
    json.dump(split_record, f, indent=4)
    
print("Splits successfully saved to JSON!")


Splits successfully saved to JSON!


In [70]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_loss_history) + 1), train_loss_history, label="Train Loss")
plt.plot(range(1, len(val_loss_history) + 1), val_loss_history, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss 2")
plt.legend()
plt.grid(True)
plt.savefig(
    "/kaggle/working/val_vs_train.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()